# *<center> The RF Quadrupole: oscillations, mass filtering, and the Mathieu diagram </center>*

**Purpose.** The einzel notebook was electrostatics: solve once, fly.
This one adds **time**: a radio-frequency drive that confines ions with a
field that has no static minimum at all. We look at the CAD meshes in 3-D,
fly ions released *off-axis* so their oscillations show, watch two masses
meet opposite fates in the same device, ride the **pseudopotential** well
the slow motion actually lives in, map the **zone-1 Mathieu stability
diagram** by flying ions, and then interrogate the map: how much does the
scan grid matter, how much does the hold time matter, and what does the
small offset from the ideal curve *measure*?

```
PROVENANCE
  device   : examples/quadrupole_stl_rods_transport.json (2-D cross-
             section, STL rods, 2 MHz) + the full-3-D variant
  theory   : Mathieu characteristic curves computed in this notebook
             (tridiagonal eigenvalue method), self-checked against the
             known zone-1 edge q = 0.9080
  aligned  : conventions verified identical to ion_playground v25
             (quadrupole.aq_to_uv / uv_to_aq: same a, q definitions and
             the same U = a k/8, V = q k/4 inversion; round-rod ratio
             R/r0 = 1.148 per its ROD_RATIO)
  authored : Fable + CRG, 2026-07 notebook series
```

> **Parameters live next to the cell that first uses them** — adjust a
> stage's knobs right where you are, no scrolling to the top. Stage 0
> holds only truly global style and physical constants. And every
> equation comes with a plain-language "why it is here."

### The following notebook adds TIME to the electrostatics of notebook 01: a radio-frequency drive that confines ions with a field having no static minimum at all. Specific topics include:

* The RF drive model: groups, phases, and DC on top of RF.
* Why a phase must be named before any RF field can be shown.
* Off-axis flight: micromotion riding on secular motion.
* Mass filtering, flown: two m/z in one device, opposite fates.
* The pseudopotential well the slow motion actually lives in.
* The zone-1 Mathieu stability diagram, mapped by flying ions and laid
  over the textbook curves — then interrogated for what the difference
  measures.

### Conventions used in this document:

* **Units are mm, eV, microseconds and volts** unless a name says
  otherwise; every parameter carries its unit in its name or a comment.
* **CAPITALS are parameters you are meant to change**; they sit in a cell
  immediately before the stage that first uses them.
* **$a$ and $q$ are the dimensionless Mathieu parameters**; $U$ is the DC
  bias between rod pairs and $V$ the RF amplitude; $r_0$ is the axis-to-rod
  distance, **measured from the solved mask**, never typed.
* **`spec` is the declaration, `model` is the solved field.** Anything
  read off `model` is what the kernel actually flew.
* Equations are numbered (**Eqn #1**, **Eqn #2**, ...) and the code that
  implements one names it.

____

## Stage 0 — global style and constants (everything else is per-stage)

## The instrument, before any statistics

The device this notebook flies, drawn from the **solver's own electrode mask** (not a redrawing) with example ion paths exactly as flown. You are looking at the four rods with the RF field at one instant, and example ions showing the two-timescale motion: slow secular oscillation with fast micromotion riding on it.

Deck: `examples/quadrupole_stl_rods_transport.json`. A geometry figure is not decoration — if the picture and the solved model can disagree, every number below is unverifiable.

In [ ]:
# ---- import-origin guard (run me FIRST) --------------------------------
# THIS notebook belongs to a repo; it must run against THAT repo's
# ion_gym, not whatever `import ion_gym` happens to find (a pip-installed
# copy, or an old tree on PYTHONPATH). A version mismatch does not fail
# politely -- it surfaces mid-run as a confusing AttributeError on some
# API the stale copy predates. So: locate the repo from this notebook's
# working directory, put it FIRST on sys.path, then verify the imported
# package actually came from here -- and REFUSE with the remedy if not.
# GENERATED CELL: every notebook carries one identical copy, stamped from
# a single definition in the development tree. Edits made here are
# overwritten the next time the notebook is regenerated.
import sys
from pathlib import Path

# leading underscores here are DELIBERATE (charter: stated reason): this
# cell is stamped into every notebook and must not collide with or
# pollute the study's own names
_here = Path.cwd().resolve()
ROOT_GUARD = next((p for p in (_here, *_here.parents)
                   if (p / "ion_gym").is_dir() and (p / "notebooks").is_dir()),
                  None)
if ROOT_GUARD is None:
    raise RuntimeError(
        f"cannot locate the ion_gym repo at or above {_here}; start the "
        f"kernel in the repo root or in a folder inside it")
if str(ROOT_GUARD) not in sys.path:
    sys.path.insert(0, str(ROOT_GUARD))

import ion_gym
_origin = Path(ion_gym.__file__).resolve().parent
if _origin.parent != ROOT_GUARD:
    raise RuntimeError(
        f"imported ion_gym v{ion_gym.__version__} from {_origin}, which is "
        f"NOT this repo ({ROOT_GUARD / 'ion_gym'}). Either the kernel "
        f"already imported a stale copy (restart the kernel and run this "
        f"cell first) or another copy shadows the repo (a pip-installed "
        f"ion_gym: `pip uninstall ion_gym`; or a stale PYTHONPATH entry). "
        f"Refusing now beats an AttributeError several cells later.")
print(f"ion_gym v{ion_gym.__version__} · loaded from {_origin}")


<!-- origin-guard-note -->
The cell above only pins this notebook to its own repo's `ion_gym`. The notebook proper begins below.


In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/quadrupole_stl_rods_transport.json', banked='panel_quadrupole.png', height=520)


In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
# path anchor: every relative path below is REPO-ROOT-relative,
# independent of where Jupyter/VS Code set the working directory.
from ion_gym.io.paths import repo_root as _repo_root
import os as _os
_os.chdir(_repo_root())

import time
import numpy as np

from ion_gym.io.sim_spec import SimSpec
from ion_gym.io import paths
from ion_gym.physics.sim_build import build_run, build_needs_solve
from ion_gym.physics.stats import (
    compute_stats, stats_markdown, auto_transmitted_fate,
    enabled_planes, mz_of_results, FATE_NAME,
)
from ion_gym.viz.viz_core import (
    scene_from_simspec, describe_fates, interactive_panel, interactive_panels,
    cad_preview,
)
from ion_gym.viz.pe_view import compute_component
from IPython.display import display, Markdown   # imported ONCE, here

# Field-panel colormap: any plotly colorscale name ("Magma", "Blackbody",
# "Jet", "Hot", "Plasma", "Cividis"). Display preference only.
COLORMAP = "Magma"

# ---- plot size (exposed on purpose: Jupyter has no app pane to size the
# figure, so the notebook is the sizing authority). FIGSIZE is passed to
# every panel below as size=(width_px, height_px); set it to None to let
# the renderer pick a true-scale box from the data extent instead.
FIGSIZE = (820, 720)        # (width, height) in pixels
FIGSIZE_WIDE = (900, 420)   # for the long, thin axial panels

# Physical constants (CODATA).
E_CHARGE_C = 1.602176634e-19
AMU_KG     = 1.66053906892e-27

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


# ---- deck-inherited physics: visible and overridable -----------------
# The deck loaded below supplies the drive, the ion, the gas and the
# integration settings. Leave an entry None to INHERIT it from the deck;
# set one to OVERRIDE. Whatever ends up in force is printed, so this
# notebook's output always states its own operating point.
# (One namespaced dict, not loose globals: the first version used bare
# names like KE_EV and N_IONS, which collided with the parameters these
# notebooks already own -- and silently changed them.)
DECK_OVERRIDES = dict(
    rf_v=None, rf_f=None,        # confining-drive amplitude (V) / freq (Hz)
    mz_list=None, charge=None,   # e.g. [622.0] / 1
    ke_ev=None,                  # (lo, hi) eV
    source_t_k=None,             # K, thermal spread of initial velocities
    n_ions=None,                 # ions flown
    gas_on=None, gas=None,       # True/False / e.g. "N2"
    p_torr=None, gas_t_k=None,   # buffer-gas pressure (Torr) / temp (K)
    dt_ns=None, t_max_us=None,   # integration step (ns) / flight time (us)
)


#### Relevant References

* Quadrupole Mass Spectrometry and Its Applications
    * P. H. Dawson (ed.)
        * Elsevier (1976); reprinted by AIP Press (1995)
* Quadrupole Ion Trap Mass Spectrometry, 2nd ed.
    * R. E. March and J. F. J. Todd
        * Wiley (2005)
* Electromagnetic traps for charged and neutral particles (Nobel Lecture)
    * W. Paul
        * *Rev. Mod. Phys.* **62**, 531-540 (1990)
        * doi.org/10.1103/RevModPhys.62.531
* Radiofrequency Spectroscopy of Stored Ions I: Storage
    * H. G. Dehmelt
        * *Adv. At. Mol. Phys.* **3**, 53-72 (1967) — the pseudopotential
* Completely Derandomized Self-Adaptation in Evolution Strategies
    * N. Hansen and A. Ostermeier
        * *Evolutionary Computation* **9**(2), 159-195 (2001) — CMA-ES,
          used in notebook 01

____

## Stage A — the device: CAD meshes and the drive model

The rods are **STL meshes** (`stl_dir` in the spec): geometry from CAD,
voxelized onto the solve grid. Before any solving, look at what was
imported — the interactive preview below is the registration and units
check (drag to orbit, hover for the electrode name). This is the **CAD
truth**; what the solver flies is the voxel raster of it, and the gap
between those two is a real, measurable thing this notebook returns to in
the mesh-detail section.

In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
# ---- Parameters of the device -- change to suit your system --------------
SPEC_PATH  = str(ROOT / 'examples/quadrupole_stl_rods_transport.json')   # 2-D cross-sec

SPEC3_PATH = str(ROOT / 'examples/quadrupole_stl_rods_full_3-d.json')  # the assembly

spec = SimSpec.from_json(SPEC_PATH)      # load BY PATH: stl_dir resolves
print(f"loaded {spec.name!r}")
# Preview the FULL 120 mm assembly, not the transport example's 1 mm
# cross-section slabs: the 2-D solve below is a CUT of this device —
# legitimate because the rods are translationally invariant along the
# axis, which is exactly what "transport cross-section" declares.
cad_preview(SimSpec.from_json(SPEC3_PATH))
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(spec, **DECK_OVERRIDES)


**Important: the preview above and the 2-D solve below are TWO DIFFERENT
BUILDS.** Say that plainly, because a picture of one device followed by a
calculation on another is how a reader gets quietly misled — and an
earlier draft of this notebook did exactly that.

The cell below measures both from their meshes. What it finds:

| | 3-D assembly (previewed) | 2-D transport slice (solved below) |
|---|---|---|
| rod profile | circle with a **flat** — an arc is clipped away | full circle |
| rod placement | on the **diagonals** | on the **x and y axes** |
| tessellation | 26 facets around | 96 facets around |
| axial length | 120 mm | 1 mm (a cross-section) |

The clipped flats are deliberate — they are how the rods mount in the
real assembly. The transport slice is the **idealized** teaching version:
full cylinders, axis-aligned, so the textbook comparison in Stage E is
against the geometry the textbook actually describes.

Two further things make even the idealized rods look non-round in the
figures, and they are worth keeping apart from the clipping:

1. **STL tessellation** — a cylinder in a mesh file is an N-gon prism.
2. **Voxelization** — the solver rasterizes that mesh onto the grid, so
   the outline drawn in every 2-D figure is the contour of the **solved
   voxel mask**, stepped at `mm_per_gu`.

Every 2-D figure draws #2 on purpose: *display equals solve*, so you see
the metal the kernel actually flew through, staircase and all.

In [ ]:
# Measure the rod cross-section from the STL itself (least-squares circle).
from ion_gym.io.stl_resolve import resolve_stl_dir
from ion_gym.physics.build_stl import _require_trimesh    # same helper
trimesh = _require_trimesh()                              # cad_preview uses

def rod_profile(sp):
    """Measure an STL rod's cross-section: best-fit radius, facet count,
    fit residual, and the largest ANGULAR GAP in the boundary.

    The angular gap is the check that matters and the one an earlier
    draft of this notebook skipped: a clipped circle's vertices still lie
    perfectly ON a circle, so a small fit residual proves nothing by
    itself. A gap much larger than one facet step means an arc is
    missing — a flat was cut."""
    mesh = trimesh.load(str(resolve_stl_dir(sp) /
                            sp.geometry.electrodes[0].stl),
                        force="mesh", process=True)
    v = np.asarray(mesh.vertices)
    ring = v[np.isclose(v[:, 2], v[:, 2].min(), atol=1e-6)][:, :2]
    ring = ring[np.hypot(*(ring - ring.mean(0)).T) > 1e-6]   # drop fan centre
    A = np.c_[2*ring[:, 0], 2*ring[:, 1], np.ones(len(ring))]
    (cx, cy, k), *_ = np.linalg.lstsq(A, (ring**2).sum(1), rcond=None)
    R = float(np.sqrt(k + cx*cx + cy*cy))
    resid = float(np.abs(np.hypot(ring[:, 0]-cx, ring[:, 1]-cy) - R).max())
    ang = np.sort(np.degrees(np.arctan2(ring[:, 1]-cy, ring[:, 0]-cx)) % 360)
    gap = float(np.diff(np.r_[ang, ang[0] + 360]).max())
    return R, len(ring), resid, gap

def report_rods(label, sp):
    R, n, resid, gap = rod_profile(sp)
    shape = ("full circle" if gap < 3 * 360 / max(n, 1)
             else f"CLIPPED — a {gap:.0f} deg arc is missing (flat face)")
    print(f"{label:>18s}: R = {R:.3f} mm, {n:3d} facets, fit residual "
          f"{resid*1e6:5.1f} nm, largest gap {gap:5.1f} deg -> {shape}")
    return R

R_ROD_MM = report_rods("transport slice", spec)
report_rods("3-D assembly", SimSpec.from_json(SPEC3_PATH))

# Where do the rods SIT? (axes vs diagonals — measured, not assumed.)
for label, sp in (("transport slice", spec),
                  ("3-D assembly", SimSpec.from_json(SPEC3_PATH))):
    m_, *_ = build_run(sp)
    ele_ = np.asarray(m_.ele)
    h_ = float(getattr(m_, "mm_per_gu", None) or m_.h_mm)
    # A rod whose metal lands on NO grid node at this pitch has no label
    # in the mask; np.where(...).mean() would be NaN and silently poison
    # the bearing. Report the dropped rod by name instead (a thin slice
    # rasterised coarsely does exactly this -- it is a resolution fact
    # about THIS view, not an error to hide).
    present = [lab for lab in (1, 2, 3, 4) if np.any(ele_ == lab)]
    if len(present) < 4:
        missing = [lab for lab in (1, 2, 3, 4) if lab not in present]
        print(f"{label:>18s}: rods {missing} do not land on any grid "
              f"node at {h_:g} mm/gu (too thin for this pitch) -- "
              f"bearing skipped for this view")
        continue
    cs = [(np.where(ele_ == lab)[0].mean()*h_,
           np.where(ele_ == lab)[1].mean()*h_) for lab in (1, 2, 3, 4)]
    ax_c = (float(np.mean([c[0] for c in cs])),
            float(np.mean([c[1] for c in cs])))
    angs = sorted(round(float(np.degrees(np.arctan2(c[1]-ax_c[1],
                                                    c[0]-ax_c[0]))) % 360)
                  for c in cs)
    print(f"{label:>18s}: rod bearings {angs} deg from the axis "
          f"-> {'ON THE AXES' if 0 in angs or 180 in angs else 'ON THE DIAGONALS'}")

The drive:

* An `RFGroupSpec` carries a **frequency, amplitude, and phase**;
  electrodes join a group by name. `RFA` (phase 0°) holds one opposing
  rod pair and `RFB` (phase 180°) the other — "opposite rods, opposite
  polarity", said electrically.
* Each rod also has a `dc` value. RF on top of DC is the whole story: the
  applied rod potential is

**Eqn #1**

$$U_{\text{rod}}(t) \;=\; \pm\bigl(U \;-\; V\cos\Omega t\bigr).$$

**Why this equation:** it just names the two knobs the instrument has.
$V$ is how hard the RF shakes, $U$ is the steady bias between the pairs,
$\Omega = 2\pi f$ is how fast. Everything later — stability, mass
selection, oscillation frequency, well depth — is a combination of these
knobs and the ion's mass.

Near the axis the four rods make the ideal saddle field

**Eqn #2**

$$\Phi(x, y, t) \;=\; \bigl(U - V\cos\Omega t\bigr)\,
   \frac{x^2 - y^2}{r_0^{\,2}}.$$

**Why this equation:** a static field cannot hold an ion at a point —
Laplace's equation forbids a 3-D potential minimum in free space
(Earnshaw's theorem) — so the best static field is a *saddle*: squeeze in
$x$, spill in $y$. The quadrupole's trick is to **flip the saddle** at
radio frequency, faster than the ion can escape. Whether that nets out to
confinement is the stability question the Mathieu map answers.

In [ ]:
for g in spec.geometry.rf_groups:
    print(f"RF group {g.name}: {g.amplitude_v:.1f} V at "
          f"{g.frequency_hz/1e6:.1f} MHz, phase {g.phase_deg:.0f} deg")
for el in spec.geometry.electrodes:
    if el.stl:
        print(f"  {el.name}: stl={el.stl}  groups={el.rf_groups}  "
              f"dc={el.dc:+.1f} V")
F_HZ  = spec.geometry.rf_groups[0].frequency_hz
OMEGA = 2.0 * np.pi * F_HZ
T_RF_US = 1e6 / F_HZ
print(f"RF period T = {T_RF_US*1e3:.0f} ns")

### The operating point: declared here, not inherited from the file

The JSON owns the **geometry** — the STL rods, their placement, the grid,
the declared symmetry. That is the laborious part, it is shared with the
GUI, and it is not what you tune.

Everything you would actually set on an instrument — the ion population,
the RF amplitude and frequency, the rod DC, the integration — is declared
**here** and written into the loaded spec. A parameter that lives only in
a file cannot be swept, optimized or explained; and the cell prints what
the file said beside what this notebook set, so nothing is inherited
silently.

In [ ]:
# ---- Parameters of the ION SOURCE -- change to suit your system ----------
N_IONS      = 12              # ions per ensemble
MZ_LIST     = [100.0, 30.0]   # m/z values flown together (the mass filter)
KE_EV       = 0.05            # birth kinetic energy
BOX_MM      = [0.8, 0.8, 0.0] # off-axis box release (on-axis is featureless)
SEED        = 0               # reproducible ensembles

# ---- Parameters of the FLIGHT -- change to suit your system --------------
DT_NS     = 2.0     # integrator step: must resolve the RF period
T_MAX_US  = 50.0    # 100 RF periods at 2 MHz
REC_EVERY = 8       # store every n-th step (see notebook 03 on decimation)

# ---- Parameters of the DRIVE -- change to suit your system ---------------
RF_AMPLITUDE_V  = 241.34    # V, per RF group (zero-to-peak)
RF_FREQUENCY_HZ = 2.0e6
ROD_DC_V        = 0.0       # U: the DC bias between rod pairs (a = 0 here)

def load_quad(**overrides):
    """Load the geometry from JSON, then APPLY the operating point.

    Every stage below calls this, so there is one definition of "the
    quadrupole at its operating point"; keyword overrides let a stage
    change one thing visibly at the call site (the stability map sweeps
    rf_v and rod_dc this way, thousands of times, as a re-weight).
    """
    # 3-D pairing demo (Stage below) passes spec_path=SPEC3_PATH so all
    # four rods resolve; every other caller inherits SPEC_PATH.
    sp = SimSpec.from_json(overrides.get("spec_path", SPEC_PATH))
    sp.source.n_ions = int(overrides.get("n_ions", N_IONS))
    sp.source.mz_list = list(overrides.get("mz_list", MZ_LIST))
    sp.source.distribution = "box"
    sp.source.box_mm = list(overrides.get("box_mm", BOX_MM))
    sp.source.ke_lo = sp.source.ke_hi = float(overrides.get("ke_ev", KE_EV))
    sp.source.seed = int(SEED)
    if "tob_span_us" in overrides:
        sp.source.tob_span_us = float(overrides["tob_span_us"])

    sp.integration.dt_ns = float(DT_NS)
    sp.integration.t_max_us = float(overrides.get("t_max_us", T_MAX_US))
    sp.integration.rec_every = int(REC_EVERY)

    for group in sp.geometry.rf_groups:
        group.amplitude_v = float(overrides.get("rf_v", RF_AMPLITUDE_V))
        group.frequency_hz = float(RF_FREQUENCY_HZ)
    # The DC bias is ANTISYMMETRIC between the two rod pairs: +U on the
    # pair carrying RFA, -U on the pair carrying RFB. That is what makes
    # 'a' a single number instead of four.
    u_dc = float(overrides.get("rod_dc", ROD_DC_V))
    for el in sp.geometry.electrodes:
        if el.stl:
            el.dc = u_dc if "RFA" in (el.rf_groups or []) else -u_dc
    return sp

from_file = SimSpec.from_json(SPEC_PATH)
spec = load_quad()
rows = [("ions", from_file.source.n_ions, spec.source.n_ions),
        ("m/z", from_file.source.mz_list, spec.source.mz_list),
        ("release", from_file.source.distribution,
         f"{spec.source.distribution} {spec.source.box_mm}"),
        ("dt (ns)", from_file.integration.dt_ns, spec.integration.dt_ns),
        ("t_max (us)", from_file.integration.t_max_us,
         spec.integration.t_max_us),
        ("RF (V)", f"{from_file.geometry.rf_groups[0].amplitude_v:g}",
         f"{spec.geometry.rf_groups[0].amplitude_v:g}"),
        ("RF (MHz)", f"{from_file.geometry.rf_groups[0].frequency_hz/1e6:g}",
         f"{spec.geometry.rf_groups[0].frequency_hz/1e6:g}"),
        ("rod DC (V)", f"{from_file.geometry.electrodes[0].dc:g}",
         f"{spec.geometry.electrodes[0].dc:g}")]
table = ["| quantity | from the JSON | set by this notebook |", "|---|---|---|"]
table += [f"| {a} | {b} | {c} |" for a, b, c in rows]
display(Markdown("\n".join(table)))
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(from_file, **DECK_OVERRIDES)


### Declaring groups and assigning electrodes

An `RFGroupSpec` is a named waveform — amplitude, frequency, phase — and
an electrode joins it **by name**, on top of whatever DC that electrode
carries:

$$V_{\text{electrode}}(t) \;=\; V_{\text{dc}}
   \;+\; A\,\sin(2\pi f t + \varphi).$$

For a quadrupole the assignment is the physics: **opposite** rods share a
phase, **adjacent** rods are 180° apart. Get that wrong and the four rods
no longer make a quadrupole field at all — the cell below does exactly
that, deliberately, and flies it.

(The funnel notebook uses the same two primitives for a very different
device: seventeen rings in antiphase plus a `DCGroupSpec` ladder. Same
declarations, different instrument.)

In [ ]:
rows = ["| electrode | RF group | dc (V) | position (mm) |", "|---|---|---|---|"]
ele_map = np.asarray(build_run(spec)[0].ele)
h_map = float(getattr(build_run(spec)[0], "mm_per_gu", None) or 0.2)
for i, el in enumerate(spec.geometry.electrodes, start=1):
    if not el.stl:
        continue
    ii, jj = np.where(ele_map == i)
    rows.append(f"| {el.name} | {', '.join(el.rf_groups) or '-'} | "
                f"{el.dc:+.1f} | ({ii.mean()*h_map:.2f}, "
                f"{jj.mean()*h_map:.2f}) |")
display(Markdown("\n".join(rows)))

In [ ]:
# The deliberate mistake: pair ADJACENT rods instead of opposite ones.
def fly_pairing(spec_in, label):
    model_p, fly_p, cols_p, births_p = build_run(spec_in)
    kinds = [fly_p(i)[1]["kind"] for i in range(len(births_p))]
    confined = sum(1 for k in kinds if k == 2) / len(kinds)
    print(f"{label:>22s}: {confined:5.0%} of ions still confined after "
          f"{spec_in.integration.t_max_us/T_RF_US:.0f} RF periods")
    return confined

# The pairing test needs all FOUR rods, so it runs on the 3-D ASSEMBLY
# deck: the transport slice (SPEC_PATH) is a 2-rod cross-section by
# construction -- only two rods intersect its plane -- so it cannot show
# an opposite-vs-adjacent pairing. The 3-D deck resolves all four.
correct = load_quad(spec_path=SPEC3_PATH, mz_list=[100.0], n_ions=8)
groups_correct = {el.name: list(el.rf_groups)
                  for el in correct.geometry.electrodes if el.stl}

# WHICH rods are adjacent is a question about GEOMETRY, so measure it from
# the solved mask rather than trusting the numbering (rod_2 need not sit
# next to rod_1). ele_map/h_map are taken from THIS deck's own solved
# mask (not Stage B's transport-slice map, which holds only two rods).
model_pair = build_run(correct)[0]
ele_map3 = np.asarray(model_pair.ele)
h_map3 = float(getattr(model_pair, "mm_per_gu", None) or 0.5)
rod_centroids = {}
for i, el in enumerate(correct.geometry.electrodes, start=1):
    if not el.stl:
        continue
    # The 3-D assembly mask is (nx, ny, nz); the rod centroid we want is
    # in the TRANSVERSE (x, y) plane, so take the first two index arrays
    # and collapse z. (The transport slice was 2-D -- hence two indices
    # there, three here.)
    where = np.where(ele_map3 == i)
    ii, jj = where[0], where[1]
    if len(ii) == 0:
        print(f"rod {el.name!r} lands on no grid node at "
              f"{h_map3:g} mm/gu -- excluded from the pairing test")
        continue
    rod_centroids[el.name] = (ii.mean() * h_map3, jj.mean() * h_map3)
if len(rod_centroids) < 4:
    raise ValueError(
        f"only {len(rod_centroids)} of 4 rods rasterise at "
        f"{h_map3:g} mm/gu on the 3-D assembly deck -- the pairing test "
        f"needs all four. Check the deck/pitch before trusting the mask.")
# the device centre is the mean of the rod centroids (Stage B measures the
# same quantity for r0; computed locally here so this cell stands alone)
centre_x = float(np.mean([c[0] for c in rod_centroids.values()]))
centre_y = float(np.mean([c[1] for c in rod_centroids.values()]))
bearings = {name: float(np.degrees(np.arctan2(c[1] - centre_y,
                                              c[0] - centre_x))) % 360
            for name, c in rod_centroids.items()}
order = sorted(bearings, key=lambda n: bearings[n])
print("rod bearings (deg):", {n: round(bearings[n]) for n in order})

# Re-assign so that NEIGHBOURS share a phase: going around the ring,
# the first two rods on one group and the last two on the other.
swapped = load_quad(spec_path=SPEC3_PATH, mz_list=[100.0], n_ions=8)
adjacent_pairing = {order[0]: ["RFA"], order[1]: ["RFA"],
                    order[2]: ["RFB"], order[3]: ["RFB"]}
for el in swapped.geometry.electrodes:
    if el.name in adjacent_pairing:
        el.rf_groups = list(adjacent_pairing[el.name])

print("correct pairing (opposite rods share a phase):", groups_correct)
print("swapped pairing (neighbours share a phase)  :", adjacent_pairing)
print()
kept_ok = fly_pairing(correct, "opposite rods paired")
kept_bad = fly_pairing(swapped, "adjacent rods paired")
if kept_bad < kept_ok:
    print(f"\n-> mis-pairing costs {100*(kept_ok-kept_bad):.0f} points of "
          "confinement. With neighbours in phase the leading term is no "
          "longer the quadrupole saddle, so there is no restoring force "
          "proportional to displacement -- the thing every equation in "
          "Stage D and E assumes.")
else:
    print(f"\n-> at this operating point the mis-paired drive did NOT "
          f"confine less ({kept_bad:.0%} vs {kept_ok:.0%}); check the "
          "operating point before concluding anything.")

## Stage B — solve once, drive by re-weight

Same linearity as the einzel, with time attached: bases $\phi_i$ are
solved per electrode (geometry only), and the drive assembles

**Eqn #3**

$$\Phi(\mathbf{r}, t) \;=\;
   \underbrace{\textstyle\sum_i U_i\,\phi_i(\mathbf{r})}_{A\ \text{(static)}}
   \;+\;
   \underbrace{\textstyle\sum_k V_k\,\phi_{B_k}(\mathbf{r})}_{B_k}
   \cos(\Omega_k t + \varphi_k).$$

**Why this equation:** it is why the Mathieu map is affordable — every
operating point of the scan only changes weights on already-solved
fields. Hundreds of points cost seconds.

$r_0$ is **measured from the solver's own electrode mask** (the number
the ions fly in, not a number typed from a drawing), and the figures show
the electrodes as **outlines of that same mask** — one boundary line per
rod, drawn over the field.

In [ ]:
needs_solve = build_needs_solve(spec)
t0 = time.time()
model, fly, col_names, births = build_run(spec)
print(f"build_run: {time.time() - t0:.2f} s "
      f"({'cold solve + cache store' if needs_solve else 'cache hit'})")

ele = np.asarray(model.ele)
h = float(getattr(model, "mm_per_gu", None) or model.h_mm)

# THE DEVICE CENTER IS MEASURED, NOT ASSUMED. The domain center and the
# quad's own axis are different points in this assembly (the CAD sits
# ~0.12 mm off the solve box); using the domain center for r0 finds the
# distance to the CORNER of the nearest rod and biases every a, q
# conversion. The quad center is the mean of the four rod centroids —
# from the solver's own mask.
# THE LABELS COME FROM THE MASK, NOT FROM AN ASSUMPTION. Every declared
# STL electrode must actually put metal in this slice; a missing label
# means the fixture/domain placement is broken (the signed-frame
# defect was exactly this: two rods off-domain, two bases empty), and a
# "center" measured from the survivors is a wrong number wearing a
# plausible value. Refuse and name the absentees instead.
n_stl = sum(1 for e in spec.geometry.electrodes if e.stl)
want = list(range(1, 1 + n_stl))
have = sorted(int(v) for v in np.unique(ele) if 0 < v <= n_stl)
missing = [lab for lab in want if lab not in have]
if missing:
    names = [spec.geometry.electrodes[lab - 1].name for lab in missing]
    raise ValueError(
        f"electrode labels {missing} ({', '.join(names)}) have no metal "
        f"in this slice -- the declared rods are not all inside the solve "
        f"domain (see examples/quad_stl_rods/README.md on the STL frame)")
cents = []
for lab in want:
    ii, jj = np.where(ele == lab)
    cents.append((ii.mean() * h, jj.mean() * h))
CENTER_MM = (float(np.mean([c[0] for c in cents])),
             float(np.mean([c[1] for c in cents])))
faces = []
for lab, _c in zip(want, cents):
    ii, jj = np.where(ele == lab)
    faces.append(float(np.min(np.hypot(ii*h - CENTER_MM[0],
                                       jj*h - CENTER_MM[1]))))
R0_MM = float(np.mean(faces))
print(f"quad center (mean of rod centroids): "
      f"({CENTER_MM[0]:.3f}, {CENTER_MM[1]:.3f}) mm — the domain center "
      f"is ({(ele.shape[0]-1)/2*h:.3f}, {(ele.shape[1]-1)/2*h:.3f})")
print(f"per-rod nearest faces: {[f'{d:.3f}' for d in faces]} mm "
      f"-> r0 = {R0_MM:.3f} mm")
print(f"(the shipped source x0 = {spec.source.x0_mm:.3f} mm IS the CAD "
      f"center; its y0 = {spec.source.y0_mm:.3f} is +0.3 mm deliberately "
      "off-axis — the release Stage D relies on)")
print(f"\nrod radius / r0 = {R_ROD_MM:.3f}/{R0_MM:.3f} = "
      f"{R_ROD_MM/R0_MM:.3f}  (the classic best-fit-to-hyperbolic ratio is "
      "1.148; ion_playground calls it ROD_RATIO)")

### Which instant? (the phase convention, and why 0 deg looks empty)

The tracer drives each RF group as

**Eqn #4**

$$s(t) \;=\; V\,\sin(\Omega t + \varphi),$$

so **phase 0 deg is the drive's zero crossing** — at that instant every
rod sits at its DC value (zero here) and there is genuinely no RF field
in the device. Asking for `phi@0` and getting a blank panel is the code
telling the truth, not a broken figure. The saddle is at full strength a
quarter period later.

**Why this matters beyond bookkeeping:** an ion's fate depends on *when*
in the cycle it arrives — which is why the stability probe later spreads
births over half a period (`tob_span_us`) instead of launching every ion
at the same phase.

Below: the saddle at its peak (90 deg), then inverted half a cycle later
(270 deg). Squeeze in one axis, spill in the other — then swapped. That
alternation is the entire confinement mechanism.

In [ ]:
scene = scene_from_simspec(spec, model, field="phi@90")
interactive_panel(scene, "xy", colorscale=COLORMAP,
                  figsize=FIGSIZE)

In [ ]:
interactive_panel(scene_from_simspec(spec, model, field="phi@270"),
                  "xy", colorscale=COLORMAP, figsize=FIGSIZE)

...and the **field magnitude** at that same instant,
$|\mathbf{E}| = |\nabla\Phi|$. **Why show both:** $\Phi$ is what the
solver computes, but the force on an ion goes as the *gradient*, so
$|\mathbf{E}|$ is what the ion actually feels.

**Note the selector: `E@90`, not `E`.** Plain `"E"` means the magnitude
of the *static* field — and on a pure-RF device every electrode has
`dc = 0`, so that field is identically zero and the panel comes back
blank. That is the code being honest rather than helpful: there is no
static field here. The instantaneous field needs its phase named, exactly
as the potential did.

The quadrupole signature to look for: $|\mathbf{E}|$ is **zero on the
axis** and grows **linearly** outward — the restoring push proportional
to displacement, which is what makes the device a lens at all — and it is
fiercest in the rod gaps.

In [ ]:
interactive_panel(scene_from_simspec(spec, model, field="E@90"),
                  "xy", colorscale=COLORMAP, figsize=FIGSIZE)

## Stage C — `dt` against the RF period

The tracer integrates $m\,\dot{\mathbf v} = -q\nabla\Phi$ with
$\Phi$ changing every step. The rule for a fixed-step integrator in an
RF field:

**Eqn #5**

$$\Delta t \;\ll\; T_{\text{RF}}
   \qquad\text{(here } T_{\text{RF}}/\Delta t \approx 250\text{)}.$$

**Why this matters:** the integrator only sees the field at the instants
it samples; a step that is a large fraction of the cycle pushes the ion
with yesterday's saddle, numerically leaking energy in or out — a stable
orbit can look unstable, or worse, the reverse.

In [ ]:
DT_NS = float(spec.integration.dt_ns)
print(f"dt = {DT_NS:g} ns; RF period {T_RF_US*1e3:.0f} ns "
      f"-> {T_RF_US*1e3/DT_NS:.0f} steps per RF cycle")
print(f"t_max = {spec.integration.t_max_us:g} us "
      f"= {spec.integration.t_max_us/T_RF_US:.0f} RF periods")

## Stage D — off-axis flight: micromotion, secular motion, two masses

The source is a **box release** slightly off the axis — on purpose. An
ion born exactly on the axis of an ideal quadrupole feels zero field and
drifts in a dead-straight, uninteresting line; the physics lives
off-axis, where the restoring push is.

Newton's law in the saddle field, in dimensionless time
$\xi = \Omega t/2$, is the **Mathieu equation**:

**Eqn #6**

$$\frac{d^2 u}{d\xi^2} + \bigl(a_u - 2 q_u \cos 2\xi\bigr)\,u = 0,
\qquad
a = \frac{8\,e\,U}{m\,r_0^2\,\Omega^2},
\quad
q = \frac{4\,e\,V}{m\,r_0^2\,\Omega^2},$$

with $u$ either $x$ ($+a, +q$) or $y$ ($-a, -q$). **Why this equation:**
the rescaling squeezes every quadrupole ever built — any rods, voltages,
frequency, mass — into **two numbers**. Two devices at the same $(a, q)$
fly the same orbit in rescaled time; one diagram is the operating manual
for all of them; and at fixed voltages each mass sits at its *own*
$(a, q)$ — which is what makes a mass filter. (These definitions and
their inversion are verified identical to `ion_playground.quadrupole`.)

A stable orbit shows a slow, large **secular** oscillation with a fast,
small **micromotion** ripple riding on it; for small $q$,

**Eqn #7**

$$\omega_{\text{sec}} \;\approx\; \frac{\beta\,\Omega}{2},
\qquad \beta \approx \sqrt{a + q^2/2}.$$

**Why this equation:** it is the checkable prediction — the flown
trajectory has a measurable oscillation frequency, and the formula says
what it must be from the knob settings alone. Measured two cells down.

In [ ]:
spec = load_quad()          # the operating point, one definition
model, fly, col_names, births = build_run(spec)

trajectories, fates, summaries = [], [], []
t0 = time.time()
for i in range(len(births)):
    tr, s = fly(i)
    summaries.append(s)
    if tr is not None and len(tr):
        trajectories.append(tr)
        fates.append(str(s.get("kind", "")))
print(f"flew {len(trajectories)} ions ({time.time()-t0:.2f} s)\n")
for line in describe_fates(spec, model, summaries):
    print(line)
mzs = mz_of_results(spec, len(summaries))
V_OP = spec.geometry.rf_groups[0].amplitude_v
for mz in sorted(set(mzs)):
    kinds = [FATE_NAME.get(s['kind'], '?')
             for s, m_ in zip(summaries, mzs) if m_ == mz]
    q_here = 4*E_CHARGE_C*V_OP / ((mz*AMU_KG)*(R0_MM*1e-3)**2*OMEGA**2)
    print(f"m/z {mz:>5.0f} (q = {q_here:.2f}): "
          + ", ".join(f"{k}x{kinds.count(k)}" for k in dict.fromkeys(kinds)))

### Why this figure has only one view

Every panel so far is `xy` — and asking this scene for `xz` or `yz`
**refuses with an error**. That is deliberate, and it is worth
understanding rather than working around:

This example is a **2-D cross-section model**. The rods are
translationally invariant along the transport axis, so the field in
*every* plane along z is the same field; solving one plane and reusing it
is exact for this geometry, and it costs a few seconds instead of a few
minutes. That is why the teaching examples and the stability map below
are affordable at all.

The price is that the model contains **no axial information whatsoever** —
there is no z to slice. A renderer that quietly drew an `xz` panel here
would be inventing a picture of something never computed, so it refuses
and says so.

**To see ions travelling down the device, you need a 3-D solve** — that is
the closing section, where the same physics is repeated on the 120 mm
assembly and all three views are real.

In [ ]:
# The cross-section with the flown paths: bounded wiggly orbits (m/z 100,
# q = 0.41) and rod strikes (m/z 30, q = 1.36) in the same frame.
scene = scene_from_simspec(spec, model, field="phi@90",
                           trajs=trajectories, fates=fates)
interactive_panel(scene, "xy", colorscale=COLORMAP,
                  figsize=FIGSIZE)

In [ ]:
stats_table = compute_stats(summaries,
                   transmitted_fate=auto_transmitted_fate(spec),
                   mz=mzs, planes=enabled_planes(spec))
Markdown(stats_markdown(stats_table))

In [ ]:
# Secular-frequency check: FFT the slow oscillation of one confined
# m/z-100 ion and compare with beta*Omega/2.
try:
    i_stable = next(i for i, (s, m_) in enumerate(zip(summaries, mzs))
                    if m_ == 100.0 and s["kind"] == 2)
except StopIteration:
    from collections import Counter
    _fates = dict(Counter((m_, s["kind"]) for s, m_ in zip(summaries, mzs)))
    raise RuntimeError(
        "no m/z-100 ion survived to t_max (kind 2) for the FFT -- a "
        "confining operating point should hold every m/z-100 ion; "
        f"(m/z, fate) counts were {_fates}") from None
tr = trajectories[i_stable]
t_us, x_mm = tr[:, 0], tr[:, 1] - CENTER_MM[0]
dt_rec = (t_us[-1] - t_us[0]) / (len(t_us) - 1)
X = np.abs(np.fft.rfft(x_mm - x_mm.mean()))
fr = np.fft.rfftfreq(len(x_mm), d=dt_rec)          # MHz (1/us)
f_meas = fr[1 + int(np.argmax(X[1:len(X)//2]))]
q100 = 4*E_CHARGE_C*V_OP / ((100.0*AMU_KG)*(R0_MM*1e-3)**2*OMEGA**2)
beta = float(np.sqrt(q100**2 / 2.0))               # a = 0 here
f_pred = beta * (F_HZ/1e6) / 2.0
print(f"measured secular frequency : {f_meas*1e3:.0f} kHz")
print(f"predicted beta*Omega/2      : {f_pred*1e3:.0f} kHz "
      f"(beta = {beta:.3f} at q = {q100:.3f}, a = 0)")
print(f"difference {abs(f_meas-f_pred)/f_pred*100:.1f}% — FFT bin width is "
      f"{(fr[1]-fr[0])*1e3:.0f} kHz and the small-q formula is itself "
      "approximate at q = 0.41")

## The pseudopotential — the well the secular motion rides in

Average over the fast micromotion and the ion behaves as if it sat in a
*static* effective well (Dehmelt's pseudopotential):

**Eqn #8**

$$U_{\text{pseudo}}(\mathbf{r}) \;=\;
   \frac{e^2\,E_0(\mathbf{r})^2}{4\,m\,\Omega^2},$$

where $E_0$ is the local RF field amplitude. **Why this equation:** it
turns a time-dependent problem into a static picture you can reason
about. The ion is pushed harder during the half-cycle it spends in the
stronger-field region than during the half it spends in the weaker one;
the tiny per-cycle imbalance averages to a steady force toward weak
field. Since a quadrupole's $|E_0|$ is zero on the axis and grows
outward, "toward weak field" means *toward the axis*: a well. Its depth
at the rod face has the famous closed form $D = qV/4$, checked below
against the solved field. (Formula and machinery per `pe_view` — the
same code the GUI's PE tab runs — and identical to
`ion_playground.pseudopotential`.)

The PE surface is a **view you can slice**: the overlay shows the well in
the cross-section, and the lineout below it cuts through the well along a
chosen axis — the same `plane`/`index` arguments slice 3-D models.

In [ ]:
# ---- Parameters of the pseudopotential section -- change to suit your system
PE_MZ = 100.0        # the pseudopotential is ION-SPECIFIC: well depth ~ 1/m
PE_SLICE_AXIS = "y"  # lineout axis through the well ("x" or "y")

x_pe, y_pe, PE, ele_pe = compute_component(model, PE_MZ, 1, "effective")
PE = np.asarray(PE)
import plotly.graph_objects as go
fig_pe_well = go.Figure()
fig_pe_well.add_heatmap(x=np.asarray(x_pe), y=np.asarray(y_pe), z=PE.T,
                  colorscale=COLORMAP,
                  colorbar=dict(title="PE (eV)"), zmin=0.0,
                  zmax=float(np.nanpercentile(PE[np.isfinite(PE)], 99)))
fig_pe_well.update_layout(title=f"pseudopotential well, m/z {PE_MZ:g} "
                          "(effective = DC + RF Dehmelt)",
                    xaxis_title="x (mm)", yaxis_title="y (mm)",
                    width=640, height=600,
                    margin=dict(l=60, r=110, t=48, b=50))
fig_pe_well

In [ ]:
# Slice through the well ALONG BOTH AXES, through the measured center.
# Depth is read at the last vacuum node before each rod face — a probe AT
# the face reads inside metal, where E collapses and the number lies.
ci = int(np.argmin(np.abs(np.asarray(x_pe) - CENTER_MM[0])))
cj = int(np.argmin(np.abs(np.asarray(y_pe) - CENTER_MM[1])))
PE0 = float(PE[ci, cj])
q_true = 4*E_CHARGE_C*V_OP / ((PE_MZ*AMU_KG)*(R0_MM*1e-3)**2*OMEGA**2)
D_dehm = q_true * V_OP / 4.0
fig_pe_slice = go.Figure()
ele2 = np.asarray(ele_pe)
depths = []
for lab, s_mm, line, mline in (("x", np.asarray(x_pe), PE[:, cj],
                                ele2[:, cj]),
                               ("y", np.asarray(y_pe), PE[ci, :],
                                ele2[ci, :])):
    r = s_mm - (CENTER_MM[0] if lab == "x" else CENTER_MM[1])
    fig_pe_slice.add_scatter(x=r, y=line, mode="lines", name=f"along {lab}")
    for sgn in (+1, -1):
        # OUTERMOST VACUUM node per side: compute_component reports a
        # small finite PE INSIDE metal (E collapses in a conductor), so a
        # probe that walks onto a rod face reads the well as shallow —
        # exclude metal with the mask that ships alongside the PE grid,
        # then extrapolate the shoulder to r0 with the ideal r^2 law.
        sel = np.isfinite(line) & (sgn*r > 0) & (mline == 0)               & (np.abs(r) <= R0_MM + 0.5)
        if sel.any():
            i_edge = np.where(sel)[0][-1 if sgn > 0 else 0]
            r_at = abs(float(r[i_edge]))
            d_at = float(line[i_edge]) - PE0
            depths.append((f"{'+' if sgn>0 else '-'}{lab}", r_at, d_at,
                           d_at * (R0_MM / r_at)**2))
for s in (R0_MM, -R0_MM):
    fig_pe_slice.add_vline(x=s, line=dict(dash="dash", color="#9467bd"))
fig_pe_slice.update_layout(
    title=f"pseudopotential well, both axes through the measured center "
          f"(dashed = r0 = {R0_MM:.2f} mm)",
    xaxis_title="distance from center (mm)", yaxis_title="PE (eV)",
    width=800, height=360, margin=dict(l=60, r=20, t=48, b=50))
for side, r_at, d, d0 in depths:
    print(f"well shoulder {side}: {d:5.1f} eV at r = {r_at:.2f} mm "
          f"(last vacuum node) -> {d0:5.1f} eV extrapolated to r0")
D_meas = float(np.mean([d0 for *_, d0 in depths]))
print(f"mean depth at r0: {D_meas:.1f} eV vs Dehmelt qV/4 = "
      f"{D_dehm:.1f} eV at q = {q_true:.3f} "
      f"({abs(D_meas-D_dehm)/D_dehm*100:.0f}% — what remains is the round-"
      "rod field deviation, the same few-percent scale as lambda_eff)")
fig_pe_slice

## Stage E — the Mathieu stability diagram, flown

The plan: pick an $(a, q)$ grid, convert each point to real voltages by
inverting the definitions,

**Eqn #9**

$$U = \frac{a\,m\,r_0^2\,\Omega^2}{8\,e},
\qquad
V = \frac{q\,m\,r_0^2\,\Omega^2}{4\,e},$$

set them (a re-weight — milliseconds), release near-rest ions off-axis,
and ask: **still confined after the hold?** Timeout = stable.

**Why near-rest ions:** the diagram is a statement about the *equation*.
Probe ions with real transverse energy also test whether their orbit fits
the bore — that is *acceptance*, a different quantity. Near-rest release
isolates the mathematics.

**The grid is not a detail — it is the ruler.** The flown edge can only
be located to within one grid step: the map's stated answer is "the edge
lies between the last surviving column and the first failing one." Halve
the step and the bracket halves; the fine-strip section below does
exactly that where it matters.

The theory overlay: zone 1 for $a \ge 0$ is bounded by two Mathieu
**characteristic curves** — $b_1(q)$ from the $x$ equation, $-a_0(q)$
from the $y$ equation (which sees $-a$):

**Eqn #10**

$$\text{stable} \iff 0 \le a \le \min\bigl(b_1(q),\, -a_0(q)\bigr).$$

**Why these curves:** a characteristic curve is where the equation's
solutions switch from bounded oscillation to exponential growth — the
mathematical cliff edge. Each is an eigenvalue of a small tridiagonal
matrix built from the equation's Fourier series; the cell self-checks its
curves against the famous $q = 0.9080$ edge before using them.

In [ ]:
def mathieu_a0(q, n=24):
    """Characteristic value a0(q): smallest eigenvalue of the even-cosine
    Fourier matrix (standard method, Abramowitz & Stegun ch. 20).
    Notebook-local; proposed for quad_ref if a gate ever needs it."""
    d = np.array([(2*k)**2 for k in range(n)], float)
    M = np.diag(d)
    M[0, 1] = M[1, 0] = q * np.sqrt(2.0)
    for k in range(1, n - 1):
        M[k, k+1] = M[k+1, k] = q
    return float(np.linalg.eigvalsh(M)[0])

def mathieu_b1(q, n=24):
    """Characteristic value b1(q): smallest eigenvalue of the odd-sine
    Fourier matrix (basis sin((2k+1)xi))."""
    d = np.array([(2*k + 1)**2 for k in range(n)], float)
    M = np.diag(d)
    M[0, 0] -= q
    for k in range(n - 1):
        M[k, k+1] = M[k+1, k] = q
    return float(np.linalg.eigvalsh(M)[0])

qs_fine = np.linspace(0.5, 1.2, 1401)
b1_fine = np.array([mathieu_b1(q) for q in qs_fine])
Q_EDGE_TH = float(np.interp(0.0, b1_fine[::-1], qs_fine[::-1]))
print(f"computed zone-1 edge (a=0): q = {Q_EDGE_TH:.4f}  "
      f"(literature 0.9080; |d| = {abs(Q_EDGE_TH-0.9080):.4f})")
assert abs(Q_EDGE_TH - 0.9080) < 2e-3, "Mathieu curves failed self-check"

def stability_ceiling(q):
    return min(mathieu_b1(q), -mathieu_a0(q))

In [ ]:
# ---- Parameters of the Mathieu map -- change to suit your system ---------
# (THE GRID IS THE RULER — see above: the flown edge is only located to
#  within one grid step)
Q_GRID = np.arange(0.05, 1.401, 0.05)    # 28 columns; step = q resolution
A_GRID = np.arange(0.00, 0.261, 0.02)    # 14 rows;    step = a resolution
N_PER_POINT  = 4          # ions flown per (a, q) point
HOLD_PERIODS = 100        # RF periods to count as stable (see next section)
PROBE_BOX_MM = 0.8        # off-axis release box
PROBE_KE_EV  = 0.01       # near-rest: stability, not acceptance

def UV_from_aq(a, q, mz=100.0):
    # Verified identical to ion_playground.quadrupole.aq_to_uv (v25).
    k = (mz * AMU_KG) * (R0_MM * 1e-3)**2 * OMEGA**2 / E_CHARGE_C
    return a * k / 8.0, q * k / 4.0

probe = load_quad(n_ions=N_PER_POINT, mz_list=[100.0],
                  box_mm=[PROBE_BOX_MM, PROBE_BOX_MM, 0.0],
                  ke_ev=PROBE_KE_EV, tob_span_us=0.5 * T_RF_US,
                  t_max_us=HOLD_PERIODS * T_RF_US)
rfa = [el.name for el in probe.geometry.electrodes
       if el.stl and "RFA" in (el.rf_groups or [])]

def survival_at(a, q, hold_periods=None):
    """Fraction of probe ions still confined (fate = timeout) at (a, q)."""
    if hold_periods is not None:
        probe.integration.t_max_us = hold_periods * T_RF_US
    U, V = UV_from_aq(a, q)
    for g in probe.geometry.rf_groups:
        g.amplitude_v = V
    for el in probe.geometry.electrodes:
        if el.stl:
            el.dc = U if el.name in rfa else -U
    m_, f_, _, b_ = build_run(probe)               # re-weight only
    kinds = [f_(i)[1]["kind"] for i in range(len(b_))]
    return sum(1 for k in kinds if k == 2) / len(kinds)

survival = np.zeros((len(A_GRID), len(Q_GRID)))
t0 = time.time()
for iq, qv in enumerate(Q_GRID):
    for ia, av in enumerate(A_GRID):
        survival[ia, iq] = survival_at(av, qv, HOLD_PERIODS)
print(f"flew {survival.size} (a, q) points x {N_PER_POINT} ions "
      f"({HOLD_PERIODS} RF periods each) in {time.time() - t0:.0f} s")

The flown map with the theory boundary — no fitting, no scaling. One
rendering honesty note: heatmap cells are drawn **centered** on their
sample points, so a stable cell visually overhangs the line by half a
grid step even when its sample point is inside. The next section takes
the remaining, *physical* part of the offset apart.

In [ ]:
qs_th = np.linspace(0.0, 1.4, 300)
ceil_th = np.array([stability_ceiling(q) for q in qs_th])
ceil_th = np.where(ceil_th >= 0, ceil_th, np.nan)

MZ_MARKS = [30.0, 50.0, 100.0, 200.0]    # masses marked on the a = 0 axis
fig_map = go.Figure()
fig_map.add_heatmap(x=Q_GRID, y=A_GRID, z=survival, colorscale=COLORMAP,
                 zmin=0, zmax=1,
                 colorbar=dict(title=f"survival<br>({HOLD_PERIODS} periods)"))
fig_map.add_scatter(x=qs_th, y=ceil_th, mode="lines",
                 line=dict(color="#00e5ff", width=2.5),
                 name="zone-1 boundary: min(b1, -a0)")
for mz in MZ_MARKS:
    q_mz = 4*E_CHARGE_C*V_OP / ((mz*AMU_KG)*(R0_MM*1e-3)**2*OMEGA**2)
    fig_map.add_scatter(x=[q_mz], y=[0.004], mode="markers+text",
                     text=[f"m/z {mz:g}"], textposition="top center",
                     textfont=dict(size=10, color="#ffffff"),
                     marker=dict(size=9, symbol="diamond", color="#ffffff",
                                 line=dict(color="#000000", width=1)),
                     showlegend=False)
fig_map.update_layout(
    title=f"Zone-1 Mathieu diagram, FLOWN (m/z 100, r0 = {R0_MM:.2f} mm "
          f"measured, {F_HZ/1e6:.0f} MHz, {N_PER_POINT} ions/point)",
    xaxis_title="q  (RF amplitude, dimensionless)",
    yaxis_title="a  (DC bias, dimensionless)",
    width=900, height=470, margin=dict(l=60, r=20, t=50, b=50))
fig_map

## Interrogating the edge: hold time, grid, and what the offset measures

Three effects separate the flown edge from the ideal teal curve, and each
is measurable:

1. **Rendering** — half a grid step of pure drawing (cells centered on
   samples). Not physics.
2. **Finite hold** — the instability growth rate goes to **zero at the
   boundary**, so an ion just outside it diverges *slowly*: slowly enough
   to survive a short hold and be counted stable. A longer hold sharpens
   the edge. **Why that follows from the math:** the characteristic curve
   is where the growth exponent crosses zero, so points near it grow like
   $e^{\mu t}$ with tiny $\mu$ — patience is resolution.
3. **The real field** — these cylindrical rods are not ideal hyperbolas.
   (Their measured $R/r_0$ is printed in Stage B; the classic
   best-approximation ratio is $1.148$, ion_playground's `ROD_RATIO`.)
   Their effective quadrupole strength
   rescales where the true edge sits in nominal $q$. After removing 1 and
   2, what remains is a **measurement of the rod-field imperfection**:

**Eqn #11**

$$\lambda_{\text{eff}} \;=\; \frac{q_{\text{edge, theory}}}
   {q_{\text{edge, flown}}}.$$

The cell flies a **fine strip** along $a = 0$ (ten times finer than the
map) at a short and a long hold, and extracts $\lambda_{\text{eff}}$
from the long-hold edge.

In [ ]:
# ---- Parameters of the fine strip -- change to suit your system ----------
Q_STRIP    = np.arange(0.84, 1.021, 0.01)   # 10x finer than the map
HOLD_SHORT = 100                            # periods (the map's hold)
HOLD_LONG  = 400                            # patience = resolution

t0 = time.time()
survival_short_hold = np.array([survival_at(0.0, q, HOLD_SHORT) for q in Q_STRIP])
survival_long_hold  = np.array([survival_at(0.0, q, HOLD_LONG)  for q in Q_STRIP])
print(f"fine strip: {2*len(Q_STRIP)} points in {time.time()-t0:.0f} s")

def edge_of(surv):
    i = int(np.argmax(surv < 0.5))
    return float(Q_STRIP[i-1] + (Q_STRIP[i]-Q_STRIP[i-1]) *
                 (surv[i-1]-0.5)/max(surv[i-1]-surv[i], 1e-9))
q_edge_short_hold, q_edge_long_hold = edge_of(survival_short_hold), edge_of(survival_long_hold)
lambda_eff = Q_EDGE_TH / q_edge_long_hold
fig_strip = go.Figure()
fig_strip.add_scatter(x=Q_STRIP, y=survival_short_hold, mode="lines+markers",
                  name=f"hold {HOLD_SHORT} periods")
fig_strip.add_scatter(x=Q_STRIP, y=survival_long_hold, mode="lines+markers",
                  name=f"hold {HOLD_LONG} periods")
fig_strip.add_vline(x=Q_EDGE_TH, line=dict(color="#00e5ff", width=2),
                annotation_text="ideal 0.9080")
fig_strip.update_layout(title=f"a = 0 edge, fine strip: q_edge = {q_edge_short_hold:.3f} "
                          f"(short) -> {q_edge_long_hold:.3f} (long); lambda_eff = "
                          f"{lambda_eff:.4f}",
                    xaxis_title="q", yaxis_title="survival",
                    width=820, height=360,
                    margin=dict(l=60, r=20, t=48, b=50))
moved = abs(q_edge_long_hold - q_edge_short_hold) >= 0.005
print(f"flown edge: {q_edge_short_hold:.3f} at {HOLD_SHORT} periods -> {q_edge_long_hold:.3f} at "
      f"{HOLD_LONG} periods "
      + ("(a longer hold moves the edge — the finite-hold bias was real "
         "at this resolution)" if moved else
         "(unchanged at this resolution: the finite-hold bias is below "
         "0.01 in q here — the remaining offset is the FIELD, not "
         "patience)"))
print(f"lambda_eff = {Q_EDGE_TH:.4f}/{q_edge_long_hold:.3f} = {lambda_eff:.4f} — the "
      "measured effective-field scale of these cylindrical, voxelized "
      "rods relative to an ideal hyperbolic quadrupole")
fig_strip

## Mesh detail: the grid the field lives on is part of the answer

The rods above are voxelized at the spec's pitch, and that pitch is a
physics input: the **measured $r_0$** is only known to the nearest voxel,
and the stair-stepped rod surface feeds the **flown stability edge**. The
honest test is a pitch ladder — re-solve the same CAD at another exact
pitch and repeat the $a = 0$ edge measurement. (The 27.8 mm extent is 139
cells at 0.2 mm and 139 is prime, so no exact *coarser* pitch exists —
A7's integer-cell rule — and the ladder runs in the refinement
direction.) If the numbers move between rows, the coarse row was
under-resolved; if they hold, the shipped pitch is *demonstrated*
converged for these observables rather than assumed so. (This is also why the figures draw the electrode
**outline of the solved mask** rather than pretending to CAD smoothness
the solver never saw.)

In [ ]:
# ---- Parameters of the mesh-detail study -- change to suit your system ---
# Shipped vs REFINED: the 27.8 mm extent is 139 cells at 0.2 mm, and 139
# is prime, so NO exact coarser pitch exists (A7: lattice quantities are
# integer gu, exactly -- 0.4 mm would need 69.5 cells and is refused).
# Mesh detail is therefore probed by refinement; the finer row is the
# better-resolved measurement of the same instrument.
PITCHES_MM = [0.2, 0.1]     # shipped vs refined; halve h -> 4x nodes in 2-D

print(f"{'h_mm':>6s} {'nodes':>10s} {'r0_mm':>7s} {'q_edge(a=0)':>12s} "
      f"{'lambda_eff':>11s}")
rows_r0, rows_qe = [], []
for h_mm in PITCHES_MM:
    sp = load_quad(n_ions=N_PER_POINT, mz_list=[100.0],
                   box_mm=[PROBE_BOX_MM, PROBE_BOX_MM, 0.0],
                   ke_ev=PROBE_KE_EV, tob_span_us=0.5 * T_RF_US)
    sp.geometry.mm_per_gu = float(h_mm)      # the point of THIS study
    m_, f_, _, b_ = build_run(sp)
    el_ = np.asarray(m_.ele)
    # same centroid-based center/r0 measurement as Stage B — the domain
    # center is NOT the device center, at any pitch
    cs_ = []
    for lab in (1, 2, 3, 4):
        ii_, jj_ = np.where(el_ == lab)
        cs_.append((ii_.mean()*h_mm, jj_.mean()*h_mm))
    cxq_ = float(np.mean([c[0] for c in cs_]))
    cyq_ = float(np.mean([c[1] for c in cs_]))
    fs_ = []
    for lab in (1, 2, 3, 4):
        ii_, jj_ = np.where(el_ == lab)
        fs_.append(float(np.min(np.hypot(ii_*h_mm - cxq_,
                                         jj_*h_mm - cyq_))))
    r0_h = float(np.mean(fs_))
    rfa_ = [el.name for el in sp.geometry.electrodes
            if el.stl and "RFA" in (el.rf_groups or [])]
    def surv_h(q):
        k = (100.0*AMU_KG)*(r0_h*1e-3)**2*OMEGA**2/E_CHARGE_C
        V = q*k/4.0
        for g in sp.geometry.rf_groups: g.amplitude_v = V
        for el in sp.geometry.electrodes:
            if el.stl: el.dc = 0.0
        sp.integration.t_max_us = HOLD_LONG * T_RF_US
        mm_, ff_, _, bb_ = build_run(sp)
        ks = [ff_(i)[1]["kind"] for i in range(len(bb_))]
        return sum(1 for kk in ks if kk == 2)/len(ks)
    sv = np.array([surv_h(q) for q in Q_STRIP])
    i_ = int(np.argmax(sv < 0.5))
    qe = float(Q_STRIP[i_-1] + 0.01*(sv[i_-1]-0.5)/max(sv[i_-1]-sv[i_],1e-9))
    rows_r0.append(r0_h); rows_qe.append(qe)
    print(f"{h_mm:6.2f} {el_.size:10,d} {r0_h:7.2f} {qe:12.3f} "
          f"{Q_EDGE_TH/qe:11.4f}")
_moved = (abs(rows_r0[0] - rows_r0[1]) > 0.005) or (abs(rows_qe[0] - rows_qe[1]) > 0.0015)
if _moved:
    print("\nrefinement MOVED the measured r0 and/or the flown edge: the "
          "shipped pitch is under-resolved for these observables and the "
          "pitch belongs in the experiment's error budget")
else:
    print("\nrefinement left r0 and the flown edge unchanged at the printed "
          "resolution: the shipped 0.2 mm pitch is demonstrated converged "
          "for these observables (halving h is the strongest exact check "
          "this prime-length extent admits)")

## The 3-D assembly — three real views

This is the build previewed in Stage A: **clipped rods on the diagonals,
120 mm long** — a different device from the idealized cross-section
above, not a 3-D copy of it. Everything transfers except the numbers:
same drive model, same equation of motion, its own $r_0$.

Now the axial views are real, because there is a solved z to cut. Two
choices in the cell below are worth reading:

* The `xz`/`yz` cuts are taken at the **rod-centre plane**, measured from
  the mask — *not* through the beam axis. On a diagonal quadrupole, a
  plane containing the axis and parallel to x or y passes **between** all
  four rods, so it would show a perfectly correct picture of empty space.
* The field and the metal are a **cut**; the ion paths are a
  **projection** of the full 3-D trajectories onto the panel. Cutting the
  ions too would show almost nothing, since a trajectory crosses any
  given plane at a point.

The 120 mm assembly (`stl3d` route), cut in the three principal planes at
spec-derived, panel-stated positions. The axial views show what the
cross-section cannot: ions snaking down the rod gap.

In [ ]:
# ---- Parameters of the 3-D section -- change to suit your system ---------
N_IONS_3D  = 8    # (SPEC3_PATH declared in Stage A with the CAD preview)

spec_3d = SimSpec.from_json(SPEC3_PATH)
needs_solve_3d = build_needs_solve(spec_3d)
t0 = time.time()
model_3d, fly_3d, cols_3d, births_3d = build_run(spec_3d)
print(f"build: {time.time()-t0:.1f} s "
      f"({'cold solve + disk-cache store' if needs_solve_3d else 'disk-cache hit'})")
trajectories_3d, fates_3d, summaries_3d = [], [], []
for i in range(min(N_IONS_3D, len(births_3d))):
    tr, s = fly_3d(i)
    summaries_3d.append(s)
    if tr is not None and len(tr):
        trajectories_3d.append(tr)
        fates_3d.append(str(s.get("kind", "")))
for line in describe_fates(spec_3d, model_3d, summaries_3d):
    print(line)

# Cut planes MEASURED from the solved mask: the rod-centre planes (see
# above), not the beam axis — on a diagonal quad the axis plane is empty.
ele_3d = np.asarray(model_3d.ele)
h_3d = float(getattr(model_3d, "mm_per_gu", None) or model_3d.h_mm)
rod_centres_3d = [(np.where(ele_3d == lab)[0].mean() * h_3d,
                   np.where(ele_3d == lab)[1].mean() * h_3d)
                  for lab in (1, 2, 3, 4)]
axis_3d = (float(np.mean([c[0] for c in rod_centres_3d])),
           float(np.mean([c[1] for c in rod_centres_3d])))
cut_planes_3d = {
    "xy": 0.5 * spec_3d.geometry.depth_mm,        # mid-length
    "xz": float(np.mean([c[1] for c in rod_centres_3d if c[1] > axis_3d[1]])),
    "yz": float(np.mean([c[0] for c in rod_centres_3d if c[0] > axis_3d[0]])),
}
print(f"beam axis at ({axis_3d[0]:.2f}, {axis_3d[1]:.2f}) mm; "
      f"axial cuts taken at the rod-centre planes "
      f"y = {cut_planes_3d['xz']:.2f}, x = {cut_planes_3d['yz']:.2f} mm")
panels_3d = interactive_panels(
    scene_from_simspec(spec_3d, model_3d, field="phi@90",
                       trajs=trajectories_3d, fates=fates_3d, slice_at=cut_planes_3d),
    colorscale=COLORMAP, figsize=FIGSIZE_WIDE)
for v in ("xy", "xz", "yz"):
    display(panels_3d[v])
# Report what came from the deck and apply any override set above.
from ion_gym.io.deck_params import apply_deck_overrides
apply_deck_overrides(spec_3d, **DECK_OVERRIDES)


## Where the series goes next

* **Ion funnel**: DC ladders (`DCGroupSpec`) and collisional cooling —
  the `collisions` block finally turns on.
* **SLIM**: travelling-wave drives and declared mirror symmetry at scale.

`mathieu_a0`/`mathieu_b1` remain notebook-local (promotion to
`physics/quad_ref.py` proposed if a gate ever needs them). The PE slicing
used here (`compute_component`'s `plane`/`index`) applies unchanged to
3-D models.

## Read-out — what this notebook established

- **Stability is a property of (a, q), not of volts.** Two very different rod voltages that map to the same Mathieu parameters give the same trajectory shape at different absolute scales. That is why the stability diagram, not a voltage table, is the design object.
- **The secular motion is slow oscillation with fast micromotion on top.** The measured secular frequency should track ω ≈ Ω·q/(2√2) at small q; where it departs, the pseudopotential approximation is failing and only the direct RF integration is trustworthy.
- **Mass filtering is a boundary effect:** at fixed (U, V, Ω), each m/z sits at its own point in the diagram, and the filter passes exactly those inside the stable region. Scanning U and V along a line through the tip is the classic scan law.
- **What would have falsified the model:** ions surviving well outside the analytic stability boundary, or a secular frequency independent of q. Neither happens here.